<font size=10>**NETWORK**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. First Network](#3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
    
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
data = pd.read_csv('../data/preprocessed_data.csv')
# data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 30485 entries, 0 to 30484
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   idcontrato                   30485 non-null  int64  
 1   tipoContrato                 30485 non-null  str    
 2   tipoFimContrato              4090 non-null   str    
 3   CPV                          30485 non-null  str    
 4   adjudicante                  30485 non-null  str    
 5   adjudicatarios               30485 non-null  str    
 6   concorrentes                 21158 non-null  str    
 7   precoBaseProcedimento        30485 non-null  float64
 8   precoContratual              30485 non-null  float64
 9   PrecoTotalEfetivo            30485 non-null  float64
 10  dataDecisaoAdjudicacao       30485 non-null  str    
 11  dataCelebracaoContrato       30485 non-null  str    
 12  dataPublicacao               30485 non-null  str    
 13  dataFechoContrato          

In [5]:
data["dataPublicacao"] = pd.to_datetime(data["dataPublicacao"], errors='coerce')
data["dataCelebracaoContrato"] = pd.to_datetime(data["dataCelebracaoContrato"], errors='coerce')
data["dataDecisaoAdjudicacao"] = pd.to_datetime(data["dataDecisaoAdjudicacao"], errors='coerce')
data["dataFechoContrato"] = pd.to_datetime(data["dataFechoContrato"], errors='coerce')

# <font color='#BFD72F' size=6>**3. The Network**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=5>**3.1 Creating It**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [6]:
################
### BEA CODE ###
################

# # If your data is not a DataFrame, fix it
# if isinstance(data, dict):
#     data = pd.DataFrame(data)

# # Create directed graph
# G = nx.DiGraph()

# # Build graph
# for _, row in data.iterrows():
#     adjudicante = row['adjudicante']
#     adjudicatarios = row['adjudicatarios']
#     preco = row['precoContratual']
#     concorrentes = row.get('concorrentes', 0)

#     # Skip missing
#     if pd.isna(adjudicante) or pd.isna(adjudicatarios):
#         continue
#     if pd.isna(preco) or preco <= 0:
#         continue

#     log_price = np.log(preco)

#     # Handle concorrentes safely
#     if pd.isna(concorrentes):
#         concorrentes = 0

#     # Add/update nodes
#     if adjudicante not in G:
#         G.add_node(adjudicante, node_type='adjudicante')
#     else:
#         if G.nodes[adjudicante].get('node_type') == 'adjudicatario':
#             G.nodes[adjudicante]['node_type'] = 'both'

#     if adjudicatarios not in G:
#         G.add_node(adjudicatarios, node_type='adjudicatario')
#     else:
#         if G.nodes[adjudicatarios].get('node_type') == 'adjudicante':
#             G.nodes[adjudicatarios]['node_type'] = 'both'

#     # Add/update edge
#     if G.has_edge(adjudicante, adjudicatarios):
#         edge = G[adjudicante][adjudicatarios]
#         edge['weight'] += log_price
#         edge['contracts'] += 1
#         edge_conc = pd.to_numeric(edge.get('nr_concorrentes', 0), errors='coerce')
#         row_conc = pd.to_numeric(concorrentes, errors='coerce')
#         edge['nr_concorrentes'] = int(0 if pd.isna(edge_conc) else edge_conc) + int(0 if pd.isna(row_conc) else row_conc)
#     else:
#         G.add_edge(
#             adjudicante,
#             adjudicatarios,
#             weight=log_price,
#             contracts=1,
#             nr_concorrentes=concorrentes,
#             city=row.get('city', None)
#         )

In [7]:
def compute_weight(prices, mode="log_sum"):

    prices = np.array(prices)

    if mode == "sum_price":
        return prices.sum()

    elif mode == "avg_price":
        return prices.mean()

    elif mode == "log_sum":
        return np.log(prices).sum()

    elif mode == "log_mean":
        return np.log(prices).mean()

    else:
        raise ValueError(
            f"Unknown mode: {mode}. "
            f"Valid modes: sum_price, avg_price, log_sum, log_mean"
        )

In [8]:
def _ensure_dataframe(data):
    if isinstance(data, pd.DataFrame):
        return data.copy()

    if isinstance(data, dict):
        # CASE 1: dict of lists (valid dataframe)
        try:
            df = pd.DataFrame(data)
            return df
        except Exception:
            pass

        # CASE 2: dict of scalars → wrap into list
        return pd.DataFrame([data])

    raise TypeError(f"Unsupported input type: {type(data)}")

In [9]:
def build_contract_network(data, weight_mode="log_sum") -> nx.DiGraph:
    
    df = _ensure_dataframe(data)

    # --- CLEAN COLUMNS ---
    df = df.rename(columns={
        'precoContratual': 'price',
        'adjudicante': 'source',
        'adjudicatarios': 'target'
    })

    # --- BASIC CLEANING ---
    df = df.dropna(subset=['source', 'target', 'price'])
    df = df[df['price'] > 0]

    # --- CONCORRENTES ---
    if 'nr_concorrentes' in df.columns:
        df['concorrentes'] = df['nr_concorrentes']
    else:
        df['concorrentes'] = pd.to_numeric(df.get('concorrentes', 0), errors='coerce').fillna(0)

    # --- EDGE AGGREGATION ---
    edge_df = (
        df.groupby(['source', 'target'], as_index=False)
        .agg(
            total_price=('price', 'sum'),
            nr_concorrentes=('concorrentes', 'sum'),
            contracts=('price', 'count'),
            price_series=('price', list)
        )
    )

    # --- WEIGHT STRATEGY ---
    edge_df['weight'] = edge_df['price_series'].apply(
        lambda x: compute_weight(x, mode=weight_mode)
    )

    edge_df = edge_df.drop(columns=['price_series'])

    # --- BUILD GRAPH ---
    G = nx.DiGraph()

    for row in edge_df.itertuples(index=False):
        G.add_edge(
            row.source,
            row.target,
            weight=row.weight,
            total_price=row.total_price,
            nr_concorrentes=row.nr_concorrentes,
            contracts=row.contracts,
            weight_mode=weight_mode
        )

    # --- NODE TYPES ---
    sources = set(edge_df['source'])
    targets = set(edge_df['target'])

    for node in G.nodes():
        if node in sources and node in targets:
            G.nodes[node]['node_type'] = 'both'
        elif node in sources:
            G.nodes[node]['node_type'] = 'adjudicante'
        else:
            G.nodes[node]['node_type'] = 'adjudicatario'

    return G

# -------------------------------
# VISUAL ATTRIBUTES (UNIFIED)
# -------------------------------
def add_visual_attributes(
    G,
    thickness_mode="linear",
    size_mode="linear",
    scale_min=1,
    scale_max=5,
    suffix=""
):
    conc = np.array([d['nr_concorrentes'] for _, _, d in G.edges(data=True)])
    weight = np.array([d['weight'] for _, _, d in G.edges(data=True)])

    def normalize(x):
        if len(x) == 0:
            return x

        range_ = np.ptp(x)  # max - min safely

        if range_ == 0:
            return np.zeros_like(x)

        return (x - np.min(x)) / (range_ + 1e-9)

    # --- TRANSFORM ---
    if thickness_mode == "log":
        conc = np.log1p(conc)

    if size_mode == "log":
        weight = np.log1p(weight)

    conc_norm = normalize(conc)
    weight_norm = normalize(weight)

    def scale(x):
        return scale_min + (scale_max - scale_min) * x

    # --- ASSIGN ---
    for i, (u, v, d) in enumerate(G.edges(data=True)):
        d[f'edge_thickness{suffix}'] = scale(conc_norm[i])
        d[f'edge_size{suffix}'] = scale(weight_norm[i])

    return G


# -------------------------------
# RUN PIPELINE
# -------------------------------
G = build_contract_network(data, weight_mode="log_sum")

# Linear scaling
G_linear = add_visual_attributes(
    G.copy(),
    thickness_mode="linear",
    size_mode="linear",
    suffix="_linear"
)

# Log scaling
G_log = add_visual_attributes(
    G.copy(),
    thickness_mode="log",
    size_mode="log",
    suffix="_log"
)

In [10]:
weight_modes = ["sum_price", "avg_price", "log_sum", "log_mean"]

graphs = {
    mode: build_contract_network(data, weight_mode=mode)
    for mode in weight_modes
}

In [11]:
for mode, G in graphs.items():
    volume = {
        node: sum(d["total_price"] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    degree = dict(G.degree())

    corr = pd.Series(degree).corr(pd.Series(volume))

    print(f"{mode:10s} → degree-volume correlation: {corr:.3f}")

sum_price  → degree-volume correlation: 0.637
avg_price  → degree-volume correlation: 0.637
log_sum    → degree-volume correlation: 0.637
log_mean   → degree-volume correlation: 0.637


In [12]:
for mode, G in graphs.items():
    top = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:3]

    print(f"\nMode: {mode}")
    for node, deg in top:
        print(f"  {node}: {deg}")


Mode: sum_price
  Unidade Local de Saúde de São José, E. P. E.: 789
  Santa Casa da Misericórdia de Lisboa: 742
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.: 730

Mode: avg_price
  Unidade Local de Saúde de São José, E. P. E.: 789
  Santa Casa da Misericórdia de Lisboa: 742
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.: 730

Mode: log_sum
  Unidade Local de Saúde de São José, E. P. E.: 789
  Santa Casa da Misericórdia de Lisboa: 742
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.: 730

Mode: log_mean
  Unidade Local de Saúde de São José, E. P. E.: 789
  Santa Casa da Misericórdia de Lisboa: 742
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.: 730


In [13]:
rankings = {}

for mode, G in graphs.items():

    weighted_degree = {
        node: sum(d["weight"] for _, _, d in G.edges(node, data=True))
        for node in G.nodes()
    }

    # convert to ranking (higher = better rank)
    ranked = pd.Series(weighted_degree).rank(ascending=False)

    rankings[mode] = ranked

sum_price → total contract value emphasis

avg_price → typical contract size

log_sum → interaction-frequency + heavy-tail compression

log_mean → normalized interaction intensity

In [14]:
rank_df = pd.DataFrame(rankings)
rank_df.head()

,sum_price,avg_price,log_sum,log_mean
"- - Inspeção-Geral do Ministério do Trabalho, Solidariedade e Segurança Social (IGMTSSS)",1016.0,1016.0,1019.0,1019.0
eden spring portugal,5594.5,5594.5,5594.5,5594.5
"- - ADP - Águas de Portugal Internacional - Serviços Ambientais, S. A.",809.0,825.0,374.0,417.0
- - B2MOBILITY GmbH,5594.5,5594.5,5594.5,5594.5
"Aon Portugal, S.A.",5594.5,5594.5,5594.5,5594.5


In [15]:
rank_df["rank_std"] = rank_df.std(axis=1)
rank_df["rank_mean"] = rank_df.mean(axis=1)

In [16]:
# Structurally important regardless of weight strategy

stable_nodes = rank_df.sort_values("rank_std").head(10)
stable_nodes

,sum_price,avg_price,log_sum,log_mean,rank_std,rank_mean
"MArketingable, Lda",5594.5,5594.5,5594.5,5594.5,0.0,4475.6
MEO,5594.5,5594.5,5594.5,5594.5,0.0,4475.6
Enzymatic S.A.,5594.5,5594.5,5594.5,5594.5,0.0,4475.6
SERVISAN PRODUTOS HIGIENE SA,5594.5,5594.5,5594.5,5594.5,0.0,4475.6
"GRIFOLS PORTUGAL, LDA.",5594.5,5594.5,5594.5,5594.5,0.0,4475.6
"Servisan - Produtos de Higiene, S.A.",5594.5,5594.5,5594.5,5594.5,0.0,4475.6
"Grupnor Elevadores de Portugal, Lda.",5594.5,5594.5,5594.5,5594.5,0.0,4475.6
"BELTRAO COELHO, LDA.",5594.5,5594.5,5594.5,5594.5,0.0,4475.6
"ISOTOPOS E DERIVADOS (ISODER), LDA.",5594.5,5594.5,5594.5,5594.5,0.0,4475.6
"LABORATÓRIOS MEDINFAR, S. A",5594.5,5594.5,5594.5,5594.5,0.0,4475.6


<font color ='red'>These nodes:
- have identical ranks across ALL weighting schemes
- are structurally invariant in your network

They are topological anchors, not sensitive to how we measure money or interaction.

We see identical rank values like 5594.5, this suggests many tied ranks. So they are stable and saturated centrality nodes. This means that ranking resolution is too coarse and many nodes are indistinguishable under this metric.

In [17]:
# Importende depends heavily on how we measure importance

unstable_nodes = rank_df.sort_values("rank_std", ascending=False).head(10)
unstable_nodes

,sum_price,avg_price,log_sum,log_mean,rank_std,rank_mean
"Vimeca Transportes - Viação Mecânica de Carnaxide, L.da",124.0,113.0,709.0,701.0,338.661458,397.132292
"Teatro Nacional D. Maria II, E. P. E.",127.0,116.0,711.0,703.0,338.084186,399.016837
CÂMARA MUNICIPAL DE SINTRA,131.0,121.0,712.0,704.0,336.058527,400.811705
Associação de Socorros da Freguesia da Encarnação,135.0,126.0,713.0,705.0,334.033307,402.606661
"AdP Energias - Energias Renováveis e Serviços Ambientais , S. A.",882.0,899.0,310.0,322.0,331.796499,548.959300
"Conselho Diretivo do Turismo de Portugal, IP",144.0,132.0,714.0,706.0,330.296836,405.259367
"Comissão Unitária de Reformados, Pensionistas e Idosos de São João da Talha",188.0,176.0,716.0,708.0,306.052283,418.810457
Associação para tratamento das toxicodependências,196.0,184.0,717.0,709.0,302.011589,421.602318
"- - AdP Energias - Energias Renováveis e Serviços Ambientais, S.A.",927.0,933.0,405.0,451.0,290.447930,601.289586
Centro Social Paroquial de Barcarena,219.0,206.0,718.0,710.0,289.608212,428.521642


<font color='red'>These nodes change position dramatically depending on weighting scheme. Their importance depends on what we consider "importance". 

Pattern:
1. Mixed contract sizes
some large contracts
many small ones
2. irregular interaction structure
sporadic procurement behavior
3. heterogeneity in edges

sum_price -> rewards big contracts
avg_price -> smooths variability
log_sum -> rewards frequency
log_mean -> penalizes extreme dispersion

In [18]:
# # Visualize the graph
# # Layout (can take time for big graphs)
# pos = nx.spring_layout(G, k=0.15, iterations=20, seed=42)

# # Extract edge attributes
# weights = [G[u][v]['weight'] for u, v in G.edges()]
# widths = [G[u][v]['nr_concorrentes'] for u, v in G.edges()]

# # Avoid division by zero
# if len(widths) > 0 and max(widths) > 0:
#     widths = [w / max(widths) * 5 for w in widths]
# else:
#     widths = [1 for _ in widths]

# if len(weights) > 0 and max(weights) > 0:
#     weights_norm = [w / max(weights) for w in weights]
# else:
#     weights_norm = weights

# # Draw
# plt.figure(figsize=(12, 10))

# nx.draw_networkx_nodes(G, pos, node_size=50)

# nx.draw_networkx_edges(
#     G,
#     pos,
#     width=widths,                 # thickness = concorrentes
#     edge_color=weights_norm,      # color = log(price)
#     edge_cmap=plt.cm.Blues
# )

# plt.title("Network: Adjudicante → Adjudicatário")
# plt.axis('off')
# plt.show()

## <font size=5>**3.2 General Insights**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [19]:
# Number of nodes
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"\t-> Number of adjudicantes: {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicante', 'both'])}, i.e., {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicante', 'both']) / G.number_of_nodes() * 100:.2f}%")
print(f"\t-> Number of adjudicatarios: {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicatario', 'both'])}, i.e., {sum(1 for _, attr in G.nodes(data=True) if attr['node_type'] in ['adjudicatario', 'both']) / G.number_of_nodes() * 100:.2f}%")

# Number of edges
print(f"\nNumber of edges: {G.number_of_edges()}")

Number of nodes: 10163
	-> Number of adjudicantes: 1025, i.e., 10.09%
	-> Number of adjudicatarios: 9156, i.e., 90.09%

Number of edges: 19077


## <font size=5>**3.3 Rank All Entities By Degree (Number of Contracts)**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

In [20]:
def entity_type(node):
    node_type = G.nodes[node].get("node_type", "unknown")
    if node_type == "both":
        return "adjudicante and adjudicatario"
    return node_type

# Highest and lowest in-degree
highest_in = max(G.in_degree(), key=lambda x: x[1])
lowest_in = min(G.in_degree(), key=lambda x: x[1])

# Highest and lowest out-degree
highest_out = max(G.out_degree(), key=lambda x: x[1])
lowest_out = min(G.out_degree(), key=lambda x: x[1])

# Average in-degree and out-degree
in_degrees = [d for _, d in G.in_degree()]
out_degrees = [d for _, d in G.out_degree()]

avg_in = sum(in_degrees) / len(in_degrees)
avg_out = sum(out_degrees) / len(out_degrees)

print(f"Highest in-degree: {highest_in} -> {entity_type(highest_in[0])}")
print(f"Lowest out-degree: {lowest_out} -> {entity_type(lowest_out[0])}")

print(f"\nHighest out-degree: {highest_out} -> {entity_type(highest_out[0])}")
print(f"Lowest in-degree: {lowest_in} -> {entity_type(lowest_in[0])}")

print(f"\nAverage in-degree: {avg_in:.2f}")
print(f"Average out-degree: {avg_out:.2f}")

Highest in-degree: ('CLARANET II SOLUTIONS, S.A.', 63) -> adjudicatario
Lowest out-degree: ('eden spring portugal', 0) -> adjudicatario

Highest out-degree: ('Unidade Local de Saúde de São José, E. P. E.', 789) -> adjudicante
Lowest in-degree: ('- -   Inspeção-Geral do Ministério do Trabalho, Solidariedade e Segurança Social (IGMTSSS)', 0) -> adjudicante

Average in-degree: 1.88
Average out-degree: 1.88


The average in-degree and average out-degree are the same because, in any directed graph, the sum of all in-degrees equals the sum of all out-degrees, and both are equal to the number of edges.

Therefore:
  $$avg_{in}  = |E| / |V|$$
  $$avg_{out} = |E| / |V|$$

so they must be identical.

In [21]:
# Top 5 Degree Enitities
top5 = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 most connected entities:")
for entity, deg in top5:
    print(f"  {entity}: {deg}")

Top 5 most connected entities:
  Unidade Local de Saúde de São José, E. P. E.: 789
  Santa Casa da Misericórdia de Lisboa: 742
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.: 730
  Unidade Local de Saúde de Santa Maria, E. P. E.: 596
  Município de Sintra: 566


In [22]:
# Degree-1 entities
deg1 = sorted(
    [(node, degree) for node, degree in G.degree() if degree == 1],
    key=lambda x: x[0]
)

print(f"Degree-1 entities: {len(deg1)}")
for i, (entity, degree) in enumerate(deg1[:20], start=1):
    node_type = G.nodes[entity].get("node_type", "unknown")
    if node_type == "both":
        node_type = "adjudicante and adjudicatario"
    print(f"{i:>3}. {entity} (degree={degree}, type={node_type})")

if len(deg1) > 20:
    print(f"... and {len(deg1) - 20} more")

Degree-1 entities: 6073
  1. - (degree=1, type=adjudicatario)
  2. - -   Inspeção-Geral do Ministério do Trabalho, Solidariedade e Segurança Social (IGMTSSS) (degree=1, type=adjudicante)
  3. - -  Lemaitre Vascular Spain SL (degree=1, type=adjudicatario)
  4. - - 3M ESPANHA SUCURSAL PORTUGAL (degree=1, type=adjudicatario)
  5. - - 3M España, S.A. (degree=1, type=adjudicatario)
  6. - - 3M España, S.L., Sucursal em Portugal (degree=1, type=adjudicatario)
  7. - - 826133058B01 (degree=1, type=adjudicatario)
  8. - - 85765766 (degree=1, type=adjudicatario)
  9. - - A. Milne Carmo SA. (degree=1, type=adjudicatario)
 10. - - AINIA (degree=1, type=adjudicatario)
 11. - - ALBAZUL SERVICIOS INTEGRALES, S.A. (degree=1, type=adjudicatario)
 12. - - ALBAZUL SERVICOS INTEGRALES SA (degree=1, type=adjudicatario)
 13. - - ALSEAMAR (degree=1, type=adjudicatario)
 14. - - ALTEL SISTEMAS S.L. (degree=1, type=adjudicatario)
 15. - - ALTEL SISTEMAS, SL (degree=1, type=adjudicatario)
 16. - - ANDERSEN TAX

In [23]:
# All contract pairs with amounts
print("Contracts with their amounts:")
for u, v, data in G.edges(data=True):
    print(f"  {u} -- {v} : €{data['weight']:,}")

Contracts with their amounts:
  - -   Inspeção-Geral do Ministério do Trabalho, Solidariedade e Segurança Social (IGMTSSS) -- eden spring portugal : €6.393590753950631
  - - ADP - Águas de Portugal Internacional - Serviços Ambientais, S. A. -- - - B2MOBILITY GmbH : €9.972893272122906
  - - ADP - Águas de Portugal Internacional - Serviços Ambientais, S. A. -- Aon Portugal, S.A. : €10.134360356072131
  - - ADP - Águas de Portugal Internacional - Serviços Ambientais, S. A. -- INETUM ESPAÑA, S.A. - Sucursal em Portugal : €9.936154141959744
  - - ADP - Águas de Portugal Internacional - Serviços Ambientais, S. A. -- MDS – Corretor de Seguros, S.A : €8.968244318109551
  - - ADP - Águas de Portugal Internacional - Serviços Ambientais, S. A. -- Previmed – Centro de Medicina Ocupacional, Lda. : €7.568329226592811
  - - AdP Energias - Energias Renováveis e Serviços Ambientais, S.A. -- - - B2MOBILITY GmbH : €9.20515796648039
  - - AdP Energias - Energias Renováveis e Serviços Ambientais, S.A. -- A

## <font size=5>**3.4 Total Contracts Volume (Price) per Entity**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [24]:
volume = {node: 0 for node in G.nodes()}

# Step 2
for u, v, data in G.edges(data=True):
    volume[u] += data["weight"]
    volume[v] += data["weight"]

# Step 3
sorted_volume = sorted(volume.items(), key=lambda x: x[1], reverse=True)
print("Entity volume ranking:")
for entity, vol in sorted_volume:
    print(f"  {entity}: €{vol:,.0f}")

Entity volume ranking:
  Unidade Local de Saúde de São José, E. P. E.: €7,198
  Santa Casa da Misericórdia de Lisboa: €6,735
  Município de Sintra: €6,262
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.: €6,145
  Unidade Local de Saúde de Santa Maria, E. P. E.: €5,589
  Município de Oeiras: €5,050
  Centro Hospitalar Universitário Lisboa Central, E. P. E.: €4,079
  Centro Hospitalar Universitário de Lisboa Central, E.P.E. (CHULC): €3,331
  Município de Cascais: €3,133
  Unidade Local de Saúde de Lisboa Ocidental, E. P. E.: €2,781
  Unidade Local de Saúde de São José, EPE: €2,719
  Unidade Local de Saúde de Loures-Odivelas, E. P. E.: €2,716
  Gebalis - Gestão do Arrendamento da Habitação Municipal de Lisboa, E. M., S. A.: €2,698
  Guarda Nacional Republicana: €2,295
  Centro Hospitalar Universitário de Lisboa Norte, E. P. E.: €2,280
  Município de Lisboa: €2,251
  Unidade Local de Saúde de Loures-Odivelas, EPE: €2,247
  Instituto Português de Oncologia de Lisboa 

## <font size=5>**3.5 Degree VS Volume**</font> <a class="anchor" id="3.5"></a>
  
[Back to TOC](#toc)

- Degree $\rightarrow$ number of contracts associated with each entity (network degree = in_degree + out_degree)
- Volume $\rightarrow$ total contracted amount associated with each entity (sum of incident edge weights)

**Purpose**: assess which entities are most valuable and how activity (degree) relates to economic impact (volume).

Planned steps:
- compute degree and volume per node
- plot degree vs. volume (use log–log scatter), annotate top entities
- report Pearson and Spearman correlations
- fit a linear model on log-transformed values and flag outliers (high volume/low degree and high degree/low volume)

In [25]:
# Degree and volume per node
metrics = pd.DataFrame({
    "degree": pd.Series(dict(G.degree())),
    "volume": pd.Series(volume)
}).fillna(0)

In [26]:
# Keep only positive values for log-log analysis
pos = metrics[(metrics["degree"] > 0) & (metrics["volume"] > 0)].copy()
pos["log_degree"] = np.log10(pos["degree"])
pos["log_volume"] = np.log10(pos["volume"])

# Correlations on log-transformed values
pearson = pos["log_degree"].corr(pos["log_volume"], method="pearson")
spearman = pos["log_degree"].corr(pos["log_volume"], method="spearman")

print(f"Nodes used in log-log analysis: {len(pos)}")
print(f"Pearson correlation (log10):  {pearson:.4f}")
print(f"Spearman correlation (log10): {spearman:.4f}")

Nodes used in log-log analysis: 10161
Pearson correlation (log10):  0.9697
Spearman correlation (log10): 0.8734


In [27]:
# Linear model in log-log space
slope, intercept = np.polyfit(pos["log_degree"], pos["log_volume"], 1)
pos["pred_log_volume"] = intercept + slope * pos["log_degree"]
pos["residual"] = pos["log_volume"] - pos["pred_log_volume"]

print(f"\nlog10(volume) = {intercept:.4f} + {slope:.4f} * log10(degree)")


log10(volume) = 1.0241 + 0.9907 * log10(degree)


In [28]:
# Flag outliers using the box-plot rule (IQR)
degree_dict = dict(G.degree())

q1 = pd.Series(volume).quantile(0.25)
q3 = pd.Series(volume).quantile(0.75)
iqr = q3 - q1
volume_threshold = q3 + 1.5 * iqr

print(f"Volume threshold from box plot rule: €{volume_threshold:,.0f}")
print("Flagged entities (degree ≥ 3 AND volume above upper fence):")

flagged = []
for entity in G.nodes():
    if degree_dict[entity] >= 3 and volume[entity] > volume_threshold:
        flagged.append((entity, degree_dict[entity], volume[entity]))

flagged.sort(key=lambda x: x[2], reverse=True)
for entity, deg, vol in flagged:
    print(f"  {entity}  degree={deg}  volume=€{vol:,.0f}")

print(f"\nTotal flagged: {len(flagged)}")

Volume threshold from box plot rule: €46
Flagged entities (degree ≥ 3 AND volume above upper fence):
  Unidade Local de Saúde de São José, E. P. E.  degree=789  volume=€7,198
  Santa Casa da Misericórdia de Lisboa  degree=742  volume=€6,735
  Município de Sintra  degree=566  volume=€6,262
  Instituto Português de Oncologia de Lisboa Francisco Gentil, E. P. E.  degree=730  volume=€6,145
  Unidade Local de Saúde de Santa Maria, E. P. E.  degree=596  volume=€5,589
  Município de Oeiras  degree=428  volume=€5,050
  Centro Hospitalar Universitário Lisboa Central, E. P. E.  degree=450  volume=€4,079
  Centro Hospitalar Universitário de Lisboa Central, E.P.E. (CHULC)  degree=366  volume=€3,331
  Município de Cascais  degree=257  volume=€3,133
  Unidade Local de Saúde de Lisboa Ocidental, E. P. E.  degree=264  volume=€2,781
  Unidade Local de Saúde de São José, EPE  degree=285  volume=€2,719
  Unidade Local de Saúde de Loures-Odivelas, E. P. E.  degree=320  volume=€2,716
  Gebalis - Gestão do 

In [29]:
# Plotting

# prepare fit line
x_line = np.logspace(np.log10(pos["degree"].min()), np.log10(pos["degree"].max()), 200)
y_line = 10 ** (intercept + slope * np.log10(x_line))

# main traces
scatter = go.Scatter(
    x=pos["degree"],
    y=pos["volume"],
    mode="markers",
    marker=dict(size=6, color="steelblue", opacity=0.5),
    name="nodes",
    hovertemplate="%{text}<br>Degree: %{x}<br>Volume: €%{y:,.0f}",
    text=pos.index
)

fit_line = go.Scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    line=dict(color="crimson", width=2),
    name="log-log fit"
)

# annotations for top entities
annotations = []
top_entities = [entity for entity, _ in top5]

for node in top_entities:
    x = metrics.loc[node, "degree"]
    y = metrics.loc[node, "volume"]
    annotations.append(
        dict(
            x=x,
            y=y,
            text=node,
            showarrow=True,
            arrowhead=2,
            ax=10,
            ay=-10,
            font=dict(size=10),
        )
    )

fig = go.Figure(data=[scatter, fit_line])
fig.update_layout(
    title="Degree VS Volume",
    xaxis=dict(title="Degree", type="log"),
    yaxis=dict(title="Volume (€)", type="log"),
    annotations=annotations,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=60, r=20, t=60, b=60)
)

fig.update_layout(
    title="Degree vs Volume",
    
    xaxis=dict(
        title="Degree",
        type="log",
        dtick=1,              # only 10^n ticks → 1, 10, 100, 1000
        tickformat=".0f"      # show full numbers instead of scientific notation
    ),
    
    yaxis=dict(
        title="Volume (€)",
        type="log",
        dtick=1,
        # tickformat=".0f"
    ),

    annotations=annotations,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    margin=dict(l=60, r=20, t=60, b=60)
)

fig.update_xaxes(showgrid=True, gridwidth=1)
fig.update_yaxes(showgrid=True, gridwidth=1)

fig.show()

In [30]:
# TODO: add attribute adjudicante/adjudicatario to hoover in plot and maybe change the color of the point based on it 

## <font size=5>**3.6 Exporting It**</font> <a class="anchor" id="3.6"></a>
  
[Back to TOC](#toc)

In [31]:
output_path = '../graphs/gephi_graph01.gexf'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
nx.write_gexf(G, output_path)
print(f'Graph exported to {output_path}')

Graph exported to ../graphs/gephi_graph01.gexf
